## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [2]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

#### Import the CSV Data as Pandas DataFrame

In [4]:
df = pd.read_csv("./Data/after_preprocess.csv")

#### Show Top 5 Records

In [5]:
df.head()

,NAME,DIV,DEP,LOC,LEAVE DATE,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-31,CS,1.0,1,2020,4,2020-01-11,20.0,2020-02-01,1.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-01,CS,0.5,2,2020,5,2020-01-31,1.0,2020-02-15,14.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-15,AL,0.5,2,2020,5,2020-02-01,14.0,2020-02-18,3.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-18,CS,1.0,2,2020,1,2020-02-15,3.0,2020-03-10,21.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-03-10,CS,1.0,3,2020,1,2020-02-18,21.0,2020-06-11,93.0,6


In [7]:
df = df.drop(columns=['LEAVE DATE', 'PREV LEAVE DATE', 'NEXT LEAVE DATE'])
df

,NAME,DIV,DEP,LOC,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,DAYS SINCE LAST LEAVE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,1,2020,4,20.0,1.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,0.5,2,2020,5,1.0,14.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,AL,0.5,2,2020,5,14.0,3.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,2,2020,1,3.0,21.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,3,2020,1,21.0,93.0,6
...,...,...,...,...,...,...,...,...,...,...,...,...
232796,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,SK,1.0,7,2024,1,20.0,60.0,3
232797,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,DL,0.5,9,2024,5,60.0,11.0,4
232798,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,CS,1.0,10,2024,2,11.0,5.0,5
232799,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,SK,1.0,10,2024,0,5.0,1.0,6


#### Preparing X and Y variables

In [8]:
X = df.drop(columns=['DAYS UNTIL NEXT LEAVE'],axis=1)

In [9]:
X.head()

,NAME,DIV,DEP,LOC,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,DAYS SINCE LAST LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,1,2020,4,20.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,0.5,2,2020,5,1.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,AL,0.5,2,2020,5,14.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,2,2020,1,3.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,3,2020,1,21.0,6


In [12]:
y = df['DAYS UNTIL NEXT LEAVE']
y

0          1.0
1         14.0
2          3.0
3         21.0
4         93.0
          ... 
232796    60.0
232797    11.0
232798     5.0
232799     1.0
232800    37.0
Name: DAYS UNTIL NEXT LEAVE, Length: 232801, dtype: float64

#### Identifying categorical and numerical features

In [13]:
# define numerical & categorical columns
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

We have 7 numerical features : ['NO DAYS', 'LEAVE MONTH', 'LEAVE YEAR', 'LEAVE DAY OF WEEK', 'DAYS SINCE LAST LEAVE', 'DAYS UNTIL NEXT LEAVE', 'CUMULATIVE LEAVE COUNT']

We have 5 categorical features : ['NAME', 'DIV', 'DEP', 'LOC', 'TYPE']


In [14]:
print("Categories in 'NAME' variable:     ",end=" " )
print(df['NAME'].unique())

print("Categories in 'DIV' variable:  ",end=" ")
print(df['DIV'].unique())

print("Categories in 'DEP' variable:",end=" " )
print(df['DEP'].unique())

print("Categories in 'LOC' variable:     ",end=" " )
print(df['LOC'].unique())

print("Categories in 'TYPE' variable:     ",end=" " )
print(df['TYPE'].unique())

Categories in 'NAME' variable:      ['A.A.A.R.DE SILVA' 'A.A.C.P.AMARASINGHE' 'A.A.D.A.R.AMARASINGHE' ...
 'Y.P.L.R.K. DISSANAYAKE' 'Y.R.A.P.M. RAJAPAKSE' 'Y.S.D.DHARMATHILAKA']
Categories in 'DIV' variable:   ['DHA' 'DPR' 'DYA' 'DBU' 'DMD' 'DMP' 'DFI' 'DPH' 'DPJ' 'DRD' 'DCH' 'DQA'
 'DPE']
Categories in 'DEP' variable: ['SAB' 'SSO' 'SHC' 'SDE' 'SMT' 'SDM' 'SHE' 'SRC' 'SHD' 'SHT' 'SWE' 'SDS'
 'SMO' 'SEA' 'SQC' 'SSC' 'SMG' 'SMP' 'SAC' 'SIR' 'SEM' 'SIT' 'SSR' 'SSM'
 'SYD' 'SMA' 'SLE' 'SFI' 'SIN' 'SPJ' 'SPR' 'SAA' 'SBU' 'SCH' 'SHA' 'SMN'
 'SES' 'SNE' 'SQA' 'SMD' 'SRE']
Categories in 'LOC' variable:      ['SEU' 'COM' 'SWC' 'FIS' 'PPH' 'FIE' 'DOS' 'GAC' 'DFS' 'TRC' 'WTS' 'FIU'
 'PP4' 'SWR' 'MAT' 'BLP' 'KIT' 'DEO' 'MAO' 'MEW' 'SEE' 'ELW' 'ENW' 'MEC'
 'AUW' 'PFS' 'ALW' 'QCO' 'ADU' 'PPM' 'TRA' 'CAL' 'SEC' 'MPO' 'NTS' 'PEO'
 'PAY' 'BLU' 'PE1' 'SCA' 'EST' 'DMO' 'WHA' 'ITO' 'STD' 'SVG' 'WEP' 'STC'
 'FTS' 'MAS' 'KBL' 'WEL' 'SRP' 'YDU' 'SHB' 'WSO' 'LOP' 'MAR' 'PFH' 'PE2'
 'CAR' 'IMP' 'SAF' 'SHR' 'LE

#### Created column transformer pipeline

In [17]:
from sklearn.impute import SimpleImputer

# Create Column Transformer with 3 types of transformers
numerical_columns = ['NO DAYS', 'LEAVE MONTH', 'LEAVE YEAR', 'LEAVE DAY OF WEEK', 'DAYS SINCE LAST LEAVE', 'CUMULATIVE LEAVE COUNT']
categorical_columns = ['NAME', 'DIV', 'DEP', 'LOC', 'TYPE']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")), #handling missing values
        ("scaler", StandardScaler(with_mean=False))
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot_encoder", OneHotEncoder(handle_unknown='ignore')),
        ("scaler", StandardScaler(with_mean=False))
    ]
)

preprocessor = ColumnTransformer([
    ("num_pipeline", num_pipeline, numerical_columns),
    ("cat_pipeline", cat_pipeline, categorical_columns)
])

In [18]:
X = preprocessor.fit_transform(X)

In [19]:
X.shape

(232801, 1689)

#### Splitting data to test and train

In [20]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((186240, 1689), (46561, 1689))

#### Create an Evaluate Function to give all metrics after model Training

In [21]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

#### Hyperparameter Tuning 

In [25]:
from sklearn.model_selection import GridSearchCV

# Define models and parameters
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

params = {
    "Decision Tree": {
        'criterion': ['squared_error', 'friedman_mse', 'absolute_error']
    },
    "Random Forest": {
        'n_estimators': [8, 16, 32, 64, 128, 256]
    },
    "Gradient Boosting": {
        'learning_rate': [0.1, 0.01, 0.05, 0.001],
        'subsample': [0.6, 0.7, 0.75, 0.8, 0.85, 0.9],
        'n_estimators': [8, 16, 32, 64, 128, 256]
    },
    "Linear Regression": {},
    "XGBRegressor": {
        'learning_rate': [0.1, 0.01, 0.05, 0.001],
        'n_estimators': [8, 16, 32, 64, 128, 256]
    },
    "CatBoosting Regressor": {
        'depth': [6, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.6],
        'iterations': [30, 50, 100]
    },
    "AdaBoost Regressor": {
        'learning_rate': [0.1, 0.01, 0.5, 0.001],
        'n_estimators': [8, 16, 32, 64, 128, 256]
    }
}

model_list = []
best_params_list = []
r2_list = []

# Loop through each model
for model_name, model in models.items():
    print(f"Processing {model_name}...")
    param = params.get(model_name, {})  # Get params for the current model

    # Perform GridSearchCV if parameters are available
    if param:
        grid_search = GridSearchCV(estimator=model, param_grid=param, cv=3, scoring='r2', verbose=1, error_score=np.nan)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
    else:
        # Fit without GridSearchCV if no parameters are defined
        best_model = model
        best_model.fit(X_train, y_train)
        best_params = "Default parameters"

    # Evaluate the best model
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    # Append results
    model_list.append(model_name)
    best_params_list.append(best_params)
    r2_list.append(model_test_r2)
    
    # Print results
    print(f"{model_name} - Best Parameters: {best_params}")
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    print("=" * 50)

Processing Linear Regression...
Linear Regression - Best Parameters: Default parameters
Model performance for Test set
- Root Mean Squared Error: 13.3902
- Mean Absolute Error: 7.8783
- R2 Score: 0.0725
Processing Lasso...
Lasso - Best Parameters: Default parameters
Model performance for Test set
- Root Mean Squared Error: 13.7530
- Mean Absolute Error: 8.1985
- R2 Score: 0.0216
Processing Ridge...
Ridge - Best Parameters: Default parameters
Model performance for Test set
- Root Mean Squared Error: 13.3902
- Mean Absolute Error: 7.8783
- R2 Score: 0.0725
Processing K-Neighbors Regressor...


KeyboardInterrupt: 

In [24]:
lin_model = XGBRegressor(n_estimators=230, learning_rate=0.091, max_depth=18, random_state=42)
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_pred)
# score = r2_score(y_test, y_pred)*100 
print(" r2 score of the model is %.2f" %model_test_r2)
print(" mae score of the model is %.2f" %model_test_mae)

 r2 score of the model is 0.11
 mae score of the model is 7.53


In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define a custom accuracy function
def custom_date_accuracy(y_true, y_pred, tolerance=14):
    """
    Calculate the percentage of predictions within the given tolerance range.
    Args:
        y_true (array-like): True target values.
        y_pred (array-like): Predicted target values.
        tolerance (int): Number of days for the acceptable range (default is ±14 days).
    Returns:
        float: Accuracy as a percentage.
    """
    # Ensure inputs are numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate the absolute difference in days
    differences = np.abs(y_pred - y_true)
    
    # Check how many predictions fall within the tolerance range
    within_tolerance = np.sum(differences <= tolerance)
    
    # Calculate accuracy as a percentage
    accuracy = (within_tolerance / len(y_true)) * 100
    return accuracy

# Evaluate the CatBoost model with the custom accuracy metric
lin_model = XGBRegressor(n_estimators=200, learning_rate=0.091, max_depth=18, random_state=42)
lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)

# Custom accuracy
tolerance_in_days = 4  # Define the tolerance range (±14 days)
date_accuracy = custom_date_accuracy(y_test, y_pred, tolerance=tolerance_in_days)

# Print the results
print(f"Custom Date Accuracy (±{tolerance_in_days} days): {date_accuracy:.2f}%")

Custom Date Accuracy (±4 days): 39.90%
